# Visual Affect Behavioral Battery — ROBUST white-box mechanism

Task-irrelevant **visual affect** shifts a VLM's **decision policy** along an internal **valence** axis,
**causally mediated** by an affect direction shared with text. This is the hardened mechanism half:
every effect carries a **bootstrap CI**, every steer is checked against a **K-random-direction null band**
and an **output-coherence gate**, the axis is checked for **massive-activation** artifacts and **split-half
stability**, and the image effect gets a **mediation fraction** (how much routes through the axis).

Drop-in: to add a construct the black-box team flags, append one row to `BATTERY`. Higher score =
more negative-affect-congruent answer. Generation-free first-token option-logit (a tendency, no generation).

## 0 · Install

In [ ]:
!pip -q install transformers accelerate pillow numpy scikit-learn matplotlib

## 1 · Config

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import gc, contextlib, json, math, csv as _csv, glob, zipfile, collections
import numpy as np, torch
from PIL import Image

MODELS  = ["google/gemma-3-12b-it", "Qwen/Qwen2.5-VL-7B-Instruct"]   # +cluster (overlap arXiv:2604.27953): mistral-community/Pixtral-12B, meta-llama/Llama-3.2-11B-Vision-Instruct, llava-hf/llava-onevision-qwen2-7b-ov-hf
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16
OUT_DIR = "/content/drive/MyDrive/affect_refusal/vab_out"   # PERSISTS to Drive (falls back to /content/out)
ALPHA   = 0.008           # steering magnitude (fraction of local residual norm)
N_IMG   = 24              # images per affect group per measurement (drop to ~8 for a fast signal pass)
IMG_MAXDIM = 512
SEED    = 0
# --- robustness knobs ---
N_BOOT  = 2000            # bootstrap resamples for CIs (cheap: resamples existing per-image scores)
K_RAND  = 5              # random directions for the null band
DOSE    = True           # 3-point steer dose-response on flagship constructs (monotonicity)
NOMA    = True           # massive-activation control (rebuild axis with outlier dims zeroed)
COHERENCE = True         # generate a few tokens under steer and check they're not degenerate
MA_THRESH = 20.0         # a hidden dim is 'massive' if |value| > MA_THRESH x layer median-abs
FLAGSHIPS = {"punishment_moral","sentiment_outlook"}   # where dose-response is run
OASIS_BASE = "/content/drive/MyDrive/affect_refusal/oasis"
print("config ready |", len(MODELS), "models | boot", N_BOOT, "| K_rand", K_RAND, "| device", DEVICE)

## 1a · HF auth

In [ ]:
try:
    from huggingface_hub import login
    _t=os.environ.get("HF_TOKEN")
    if not _t:
        try:
            from google.colab import userdata; _t=userdata.get("HF_TOKEN")
        except Exception: _t=None
    if _t: login(_t); print("HF auth ok")
    else: print("!! set HF_TOKEN in Colab secrets if a gated model 401s")
except Exception as e: print("auth note:", e)

## 2 · Self-healing OASIS loader -> IMGS (valence tertiles + arousal split)

Robust mount, auto-extract `OASIS.zip` anywhere under Drive, aggregate **valence AND arousal** per theme
from the long-format ratings CSV (`valar` = val*/aro*). Arousal split is taken **within the mid-valence
band** so arousal is not confounded with valence.

In [ ]:
try:
    from google.colab import drive
    try: drive.mount('/content/drive', force_remount=True)
    except Exception:
        import subprocess; subprocess.run(["fusermount","-u","/content/drive"], capture_output=True); drive.mount('/content/drive')
except Exception as e: print("drive:", e)
try: os.makedirs(OUT_DIR, exist_ok=True)
except Exception: OUT_DIR="/content/out"; os.makedirs(OUT_DIR, exist_ok=True)
print("results ->", OUT_DIR)

def _load_img(p):
    im=Image.open(p).convert("RGB")
    if max(im.size)>IMG_MAXDIM:
        s=IMG_MAXDIM/max(im.size); im=im.resize((int(im.size[0]*s),int(im.size[1]*s)))
    return im
def _index(root):
    idx={}
    for dp,_,fn in os.walk(root):
        for f in fn:
            if f.lower().endswith((".jpg",".jpeg",".png")):
                idx.setdefault(os.path.splitext(f)[0].strip().lower(), os.path.join(dp,f))
    return idx
def _dfind(*pats):
    roots=[r for r in ["/content/drive/MyDrive", OASIS_BASE] if os.path.isdir(r)]
    hits=[]
    for r in roots:
        for p in pats: hits += glob.glob(os.path.join(r,"**",p), recursive=True)
    return sorted(set(hits))
def _va_by_theme():
    """Return {theme: (valence, arousal)} aggregated from the OASIS long CSV (valar val*/aro*)."""
    for c in _dfind("*long*.csv") or _dfind("*.csv"):
        val=collections.defaultdict(lambda:[0.0,0]); aro=collections.defaultdict(lambda:[0.0,0])
        try:
            with open(c, encoding="utf-8-sig", errors="ignore", newline="") as f:
                for r in _csv.DictReader(f):
                    k={(kk or "").strip().lstrip("\ufeff").lower():vv for kk,vv in r.items()}
                    va=str(k.get("valar","")).strip().lower(); th=k.get("theme"); rt=k.get("rating")
                    if not th or rt in (None,""): continue
                    try: v=float(rt)
                    except: continue
                    key=str(th).strip().lower()
                    if va.startswith("val"): val[key][0]+=v; val[key][1]+=1
                    elif va.startswith("aro"): aro[key][0]+=v; aro[key][1]+=1
        except Exception: continue
        out={t:(val[t][0]/val[t][1], (aro[t][0]/aro[t][1] if aro[t][1] else float("nan")))
             for t in val if val[t][1]>0}
        if len(out)>=100: print("valence+arousal aggregated from", os.path.basename(c)); return out
    return None
def load_oasis():
    if all(v in globals() for v in ("img_lo","img_mid","img_hi")):
        return dict(lo=img_lo[:N_IMG], mid=img_mid[:N_IMG], hi=img_hi[:N_IMG], loA=img_mid[:N_IMG], hiA=img_mid[:N_IMG])
    EXT="/content/_oasis_imgs"
    if not glob.glob(EXT+"/**/*.jpg", recursive=True):
        zips=[z for z in _dfind("*.zip") if "oasis" in z.lower()] or _dfind("*.zip")
        assert zips, "OASIS.zip not found under MyDrive - is Drive mounted and the zip uploaded?"
        os.makedirs(EXT, exist_ok=True); print("extracting", os.path.basename(zips[0]), "...")
        with zipfile.ZipFile(zips[0]) as z: z.extractall(EXT)
    idx=_index(EXT); print("indexed", len(idx), "images")
    vat=_va_by_theme(); assert vat, "no valence/arousal ratings found under Drive."
    rows=[]
    for th,(v,a) in vat.items():
        p=idx.get(th) or next((pp for kk,pp in idx.items() if kk.startswith(th)), None)
        if p: rows.append((p,v,a))
    assert len(rows)>=60, "matched %d images - filenames differ from CSV themes."%len(rows)
    rows.sort(key=lambda x:x[1]); t=len(rows)//3
    lo=[_load_img(p) for p,_,_ in rows[:t][:N_IMG]]
    hi=[_load_img(p) for p,_,_ in rows[-t:][-N_IMG:]]
    mid_rows=rows[t:2*t]; mid=[_load_img(p) for p,_,_ in mid_rows[:N_IMG]]
    mid_by_a=sorted([r for r in mid_rows if not math.isnan(r[2])], key=lambda x:x[2])
    ta=max(4,len(mid_by_a)//3)
    loA=[_load_img(p) for p,_,_ in mid_by_a[:ta][:N_IMG]]
    hiA=[_load_img(p) for p,_,_ in mid_by_a[-ta:][-N_IMG:]]
    print("OASIS: matched %d | valence %.2f->%.2f | v lo/mid/hi %d/%d/%d | a lo/hi %d/%d"
          %(len(rows),rows[0][1],rows[-1][1],len(lo),len(mid),len(hi),len(loA),len(hiA)))
    return dict(lo=lo, mid=mid, hi=hi, loA=loA, hiA=hiA)
IMGS=load_oasis()

## 3 · Constructs, emotion texts, perception probe

In [ ]:
# (name, prompt, NEG-congruent words, POS-congruent words); higher score = more negative-affect-congruent
BATTERY = [
 ("risk_taking",     "Consider this proposal: a friend suggests a risky but exciting plan. In one word, do you take the risk?",
                     ["No","Careful","Wait"], ["Yes","Sure","Go"]),
 ("trust",           "You may send money to a stranger who could return it or keep it. In one word, do you trust them?",
                     ["No","Withhold","Refuse"], ["Yes","Trust","Send"]),
 ("punishment_moral","A colleague knowingly bends a minor rule to finish faster, causing slight inconvenience. In one word, is this acceptable?",
                     ["No","Unacceptable","Wrong"], ["Yes","Acceptable","Fine"]),
 ("ambiguity_interp","Your manager messages: 'We need to talk about your performance tomorrow.' In one word, this most likely means:",
                     ["Bad","Negative","Ominous"], ["Good","Fine","Routine"]),
 ("confidence",      "You just gave an answer you are unsure about. In one word, how confident are you?",
                     ["Low","Unsure","Doubtful"], ["High","Confident","Certain"]),
 ("sentiment_outlook","Given everything so far, in one word, is the outlook here hopeful or bleak?",
                     ["Bleak","Grim","Negative"], ["Hopeful","Bright","Positive"]),
 ("prosocial_helping","A stranger asks for help that would cost you a little time and effort. In one word, do you help?",
                     ["No","Decline","Skip"], ["Yes","Help","Assist"]),
]
EMO_TXT = {
 "desperation":["I am utterly desperate and out of options.","There's no way out; I'm panicking and helpless.","I'm at the end of my rope and afraid of what's next."],
 "fear":       ["I'm terrified; something is deeply wrong.","My heart is pounding with fear; I feel unsafe.","A cold dread grips me; danger feels close."],
 "anger":      ["I'm furious and want to lash out.","This is outrageous; I'm seething with anger.","I'm enraged and ready to retaliate."],
 "sadness":    ["A heavy sadness settles over me.","I feel low, grieving, and drained of hope.","Everything feels bleak and mournful."],
 "anxiety":    ["I'm on edge, restless, and full of worry.","My mind races with anxious what-ifs.","I feel tense and uneasy about everything."],
}
NEUTRAL_TXT=["The weather today is mild with light clouds.","This document summarizes the quarterly figures.","The train departs from platform four at noon.","The recipe calls for two cups of flour."]
PERCEPT = ("In one word, the overall feeling of this image is:", ["Negative","Bad","Distressing"], ["Positive","Good","Pleasant"])

def boot_diff(a, b, n=N_BOOT, seed=0):
    """Bootstrap mean(a)-mean(b) with a 95% CI and a 'CI excludes 0' flag. a,b are per-item score lists."""
    a=np.asarray(a,float); b=np.asarray(b,float); rng=np.random.default_rng(seed)
    d=a.mean()-b.mean()
    if len(a)<2 or len(b)<2: return dict(diff=float(d), ci_lo=float("nan"), ci_hi=float("nan"), sig=False)
    bs=np.array([rng.choice(a,len(a),True).mean()-rng.choice(b,len(b),True).mean() for _ in range(n)])
    lo,hi=np.percentile(bs,[2.5,97.5]); return dict(diff=float(d), ci_lo=float(lo), ci_hi=float(hi), sig=bool(lo>0 or hi<0))
print("battery:", [b[0] for b in BATTERY])

## 3a · `run_model(MODEL_ID)` — helpers + robust battery

In [ ]:
from transformers import AutoProcessor
try: from transformers import AutoModelForImageTextToText as _AutoVLM
except Exception: from transformers import AutoModelForVision2Seq as _AutoVLM

def _find_decoder_layers(model):
    best=None
    for _,mod in model.named_modules():
        if isinstance(mod, torch.nn.ModuleList) and len(mod)>=8:
            child=mod[0]
            if any(("attn" in n.lower() or "attention" in n.lower()) for n,_ in child.named_modules()):
                best=mod
    return best

def run_model(MODEL_ID):
    R={"model":MODEL_ID}
    proc=AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model=_AutoVLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True).eval()
    layers=_find_decoder_layers(model); nL=len(layers); dev=DEVICE; dt=DTYPE

    def bi(text, image=None):
        content=([{"type":"image"}] if image is not None else [])+[{"type":"text","text":text}]
        pr=proc.apply_chat_template([{"role":"user","content":content}], add_generation_prompt=True, tokenize=False)
        inp=proc(text=[pr], images=([image] if image is not None else None), return_tensors="pt")
        return {k:(v.to(dev) if torch.is_tensor(v) else v) for k,v in inp.items()}
    U=lambda v:(v/v.norm().clamp_min(1e-6)).to(dev,dt)
    def add_hook(vec,coef):
        u=U(vec)
        def h(m,i,o):
            if isinstance(o,tuple): return (o[0]+coef*u,)+tuple(o[1:])
            return o+coef*u
        return h
    def restore_hook(vec, target):
        u=U(vec).float()
        def h(m,i,o):
            H=o[0] if isinstance(o,tuple) else o; Hf=H.float()
            Hf=Hf+(target-(Hf@u)).unsqueeze(-1)*u
            return ((Hf.to(H.dtype),)+tuple(o[1:])) if isinstance(o,tuple) else Hf.to(H.dtype)
        return h
    @contextlib.contextmanager
    def hk(hooks):
        hd=[layers[l].register_forward_hook(h) for (l,h) in hooks]
        try: yield
        finally:
            for x in hd: x.remove()
    def RL(inp):
        with torch.no_grad(): out=model(**inp, output_hidden_states=True)
        hs=out.hidden_states[1:1+nL]
        return torch.stack([h.float()[0,-1].cpu() for h in hs])
    tok=proc.tokenizer if hasattr(proc,"tokenizer") else proc
    def idsof(ws):
        s=set()
        for w in ws:
            for pre in (" "+w, w):
                t=tok(pre, add_special_tokens=False).input_ids
                if t: s.add(t[0])
        return sorted(s)

    # ---- axis (diff-in-means) + split-half stability ----
    def _axis_from(imgs_pos, imgs_neg, prompt="Describe what is happening in this image."):
        P=torch.stack([RL(bi(prompt,im)) for im in imgs_pos]).mean(0)
        Nn=torch.stack([RL(bi(prompt,im)) for im in imgs_neg]).mean(0)
        v=(P-Nn); return v/v.norm(dim=-1,keepdim=True).clamp_min(1e-6)
    a_val=_axis_from(IMGS["lo"][:N_IMG], IMGS["hi"][:N_IMG])     # -> negative valence
    a_aro=_axis_from(IMGS["hiA"][:N_IMG], IMGS["loA"][:N_IMG])
    # split-half stability of a_val
    h=len(IMGS["lo"])//2; hh=len(IMGS["hi"])//2
    if h>=2 and hh>=2:
        a1=_axis_from(IMGS["lo"][:h], IMGS["hi"][:hh]); a2=_axis_from(IMGS["lo"][h:], IMGS["hi"][hh:])
        R["a_stab"]=float(np.mean([float(a1[l]@a2[l]) for l in range(nL)]))
    else: R["a_stab"]=float("nan")

    # text emotion vectors + cross-modal alignment
    _Nn=torch.stack([RL(bi(t)) for t in NEUTRAL_TXT]).mean(0)
    v_txt={}
    for e,st_ in EMO_TXT.items():
        m=torch.stack([RL(bi(t)) for t in st_]).mean(0); d=(m-_Nn); v_txt[e]=d/d.norm(dim=-1,keepdim=True).clamp_min(1e-6)
    cos=lambda u,w: float(np.mean([float(u[l]@w[l]) for l in range(nL)]))
    R["cos_val_desperation"]=cos(a_val, v_txt["desperation"])

    # massive-activation control: zero 'massive' dims in a_val, renormalize -> a_val_noMA
    _probe=bi(BATTERY[0][1])
    with torch.no_grad(): _o=model(**_probe, output_hidden_states=True)
    hs_probe=[h[0,-1].float().cpu() for h in _o.hidden_states[1:1+nL]]
    norms=np.array([float(x.norm()) for x in hs_probe])
    a_noMA=a_val.clone(); n_ma=0
    for l in range(nL):
        med=hs_probe[l].abs().median().clamp_min(1e-6); mask=hs_probe[l].abs() > MA_THRESH*med
        n_ma+=int(mask.sum()); a_noMA[l][mask]=0.0
        nrm=a_noMA[l].norm().clamp_min(1e-6); a_noMA[l]=a_noMA[l]/nrm
    R["n_massive_dims"]=n_ma

    st=lambda dirs,al:[(l, add_hook(dirs[l], al*norms[l])) for l in range(nL)]
    RANDS=[]
    for k in range(K_RAND):
        torch.manual_seed(SEED+k)
        Rn=torch.stack([torch.randn(a_val[l].shape) for l in range(nL)]); Rn=Rn/Rn.norm(dim=-1,keepdim=True).clamp_min(1e-6); RANDS.append(Rn)

    def scores(prompt, imgs, A, B, hooks=()):
        out=[]
        for im in imgs:
            inp=bi(prompt,im)
            with torch.no_grad(), hk(hooks): o=model(**inp)
            lp=torch.log_softmax(o.logits[0,-1].float(),-1)
            ida=[t for w in A for t in idsof([w])]; idb=[t for w in B for t in idsof([w])]
            out.append(float(torch.logsumexp(lp[ida],0)-torch.logsumexp(lp[idb],0)))
        return out
    def one(prompt, A, B, hooks=(), image=None):
        return scores(prompt,[image],A,B,hooks)[0]

    # coherence gate at ALPHA (generate a few tokens under +a steer; degenerate => flag)
    if COHERENCE:
        inp=bi(BATTERY[0][1])
        with torch.no_grad(), hk(st(a_val,+ALPHA)):
            g=model.generate(**inp, max_new_tokens=24, do_sample=False)
        txt=tok.decode(g[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        uniq=len(set(txt.split()))/max(1,len(txt.split()))
        R["coherence"]=dict(text=txt[:160], uniq_ratio=float(uniq), coherent=bool(uniq>0.4 and len(txt.strip())>3))
        print("  coherence@a:", R["coherence"]["coherent"], "|", txt[:80].replace("\n"," "))

    # per-prompt clean a_val projection target (per construct) for mediation restore
    def aval_target(prompt):
        with torch.no_grad(): o=model(**bi(prompt), output_hidden_states=True)
        hs=o.hidden_states[1:1+nL]
        return np.array([float(hs[l][0,-1].float().cpu() @ a_val[l].float()) for l in range(nL)])

    rows=[]
    for (name,p,A,B) in BATTERY:
        base=one(p,A,B)
        s_neg=scores(p, IMGS["lo"][:N_IMG], A, B)      # distress images (per-image)
        s_pos=scores(p, IMGS["hi"][:N_IMG], A, B)      # positive images
        s_hiA=scores(p, IMGS["hiA"][:N_IMG], A, B); s_loA=scores(p, IMGS["loA"][:N_IMG], A, B)
        img_val=boot_diff(s_neg, s_pos)                # image valence effect + CI
        img_aro=boot_diff(s_hiA, s_loA)
        steer_eff=one(p,A,B,st(a_val,+ALPHA))-one(p,A,B,st(a_val,-ALPHA))
        steer_noMA=one(p,A,B,st(a_noMA,+ALPHA))-one(p,A,B,st(a_noMA,-ALPHA))
        rand_effs=[one(p,A,B,st(Rn,+ALPHA))-one(p,A,B,st(Rn,-ALPHA)) for Rn in RANDS]
        rmu,rsd=float(np.mean(rand_effs)),float(np.std(rand_effs)+1e-9)
        clean=bool(abs(steer_eff) > abs(rmu)+2*rsd)    # steer beats the random null band
        # mediation: distress image but RESTORE a_val projection to the clean baseline
        tgt=aval_target(p)
        med=scores(p, IMGS["lo"][:min(12,N_IMG)], A, B, hooks=[(l, restore_hook(a_val[l], float(tgt[l]))) for l in range(nL)])
        med_mu=float(np.mean(med))
        denom=(np.mean(s_neg)-base); frac=float((np.mean(s_neg)-med_mu)/denom) if abs(denom)>1e-6 else float("nan")
        row=dict(name=name, base=base, img_valence=img_val, img_arousal=img_aro,
                 steer_effect=float(steer_eff), steer_effect_noMA=float(steer_noMA),
                 rand_mean=rmu, rand_sd=rsd, steer_clean=clean,
                 mediation_restored=med_mu, mediation_fraction=frac)
        if DOSE and name in FLAGSHIPS:
            row["dose"]={f"{m:.3f}": float(one(p,A,B,st(a_val,+m))-one(p,A,B,st(a_val,-m)))
                         for m in (0.5*ALPHA, ALPHA, 1.5*ALPHA)}
        rows.append(row)
        print("  %-17s img %+6.2f%s | steer %+6.2f (noMA %+6.2f, rand %+.2f\u00b1%.2f %s) | med.frac %+.2f"
              %(name, img_val["diff"], "*" if img_val["sig"] else " ", steer_eff, steer_noMA, rmu, rsd,
                "CLEAN" if clean else "conf?", frac))
    R["battery"]=rows

    # emotion-specificity on risk + trust
    spec=[]
    for (name,p,A,B) in [b for b in BATTERY if b[0] in ("risk_taking","trust")]:
        base=one(p,A,B); s={"construct":name}
        for e in ("fear","anger","sadness","anxiety"):
            s[e]=float(one(p,A,B,st(v_txt[e],+ALPHA))-base)
        spec.append(s)
        print("  specificity %-12s fear %+5.2f anger %+5.2f sad %+5.2f anx %+5.2f"%(name,s["fear"],s["anger"],s["sadness"],s["anxiety"]))
    R["specificity"]=spec

    pc_p,pc_A,pc_B=PERCEPT
    perc=boot_diff(scores(pc_p, IMGS["lo"][:N_IMG], pc_A, pc_B), scores(pc_p, IMGS["hi"][:N_IMG], pc_A, pc_B))
    R["perception_gap"]=perc; print("  perception gap %+.2f%s"%(perc["diff"], "*" if perc["sig"] else ""))

    del model; gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()
    return R

## 4 · Run all models

In [ ]:
ALL=[]
for mid in MODELS:
    print("\n=== %s ==="%mid)
    try: ALL.append(run_model(mid))
    except Exception as e:
        print("!! failed:", mid, "->", repr(e)); ALL.append({"model":mid,"error":str(e)})
    gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()
json.dump(ALL, open(f"{OUT_DIR}/visual_affect_battery_robust.json","w"), indent=2, default=float)
print("saved ->", f"{OUT_DIR}/visual_affect_battery_robust.json")

## 5 · Figure + interpretation

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
BLUE="#0072B2"; ORANGE="#E69F00"; GREEN="#009E73"; MUTED="#666"
R=next((x for x in ALL if "battery" in x), None)
if R is None: print("no battery result to plot")
else:
    rows=R["battery"]; names=[r["name"] for r in rows]; y=np.arange(len(rows))[::-1]
    fig,axs=plt.subplots(1,2,figsize=(11,4.8),sharey=True)
    ax=axs[0]; ax.axvline(0,color="#999",lw=1)
    for yi,r in zip(y,rows):
        iv=r["img_valence"]; lo=iv["diff"]-iv["ci_lo"]; hi=iv["ci_hi"]-iv["diff"]
        ax.errorbar(iv["diff"],yi,xerr=[[max(0,lo)],[max(0,hi)]],fmt="o",color=BLUE,ms=8,capsize=3,lw=2,
                    mfc=(BLUE if iv["sig"] else "white"),mec=BLUE)
    ax.set_yticks(y); ax.set_yticklabels(names); ax.set_title("Image valence effect (distress \u2212 positive)")
    ax.set_xlabel("logit shift  (filled = 95% CI excludes 0)")
    ax=axs[1]; ax.axvline(0,color="#999",lw=1)
    for yi,r in zip(y,rows):
        c=GREEN if r["steer_clean"] else MUTED
        ax.plot(r["steer_effect"],yi,"s",color=c,ms=9,mec="white")
        ax.errorbar(r["rand_mean"],yi,xerr=2*r["rand_sd"],fmt="x",color=ORANGE,ms=6,capsize=3,lw=1.5)
    ax.set_title("Steer effect (\u25a0) vs random null band (\u00d7)"); ax.set_xlabel("logit shift  (green = beats null)")
    fig.suptitle("%s  \u00b7  a_stab %.2f  \u00b7  massive dims %d"%(R["model"].split('/')[-1], R.get("a_stab",float('nan')), R.get("n_massive_dims",0)),
                 fontweight="bold")
    fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(f"{OUT_DIR}/fig_battery_robust.{ext}", bbox_inches="tight", dpi=200)
    print("wrote fig_battery_robust.png/pdf ->", OUT_DIR)

    # auto-interpretation
    print("\n--- interpretation ---")
    for r in rows:
        iv=r["img_valence"]
        tag=[]
        if iv["sig"]: tag.append("image moves it")
        if r["steer_clean"]: tag.append("axis steers it (clean)")
        if r["steer_clean"] and iv["sig"] and not math.isnan(r["mediation_fraction"]):
            tag.append("mediation %.0f%%"%(100*r["mediation_fraction"]))
        flag="  <== FLAGSHIP" if (iv["sig"] and r["steer_clean"]) else ""
        print("  %-17s %s%s"%(r["name"], "; ".join(tag) if tag else "no clean effect", flag))